In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import optax
import numpy as np
from pyscf import gto, scf, fci
import flax.linen as nn

# 禁用 JIT 调试（正式运行时可注释掉）
# jax.config.update('jax_disable_jit', True)

# ==============================================================================
# 1. H₂ 分子与 FCI 基准
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("=" * 60)
print("FCI 基准能量 (Ha)")
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f}  |  激发能 = {(e - E_fcis[0]) * 27.2114:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# ==============================================================================
# 2. 扩展希尔伯特空间与采样器
# ==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)
hi_ext = hi ** K

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)
sampler = nk.sampler.MetropolisSampler(hi_ext, rule=single_rule, n_chains=16, sweep_size=32)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FCI 基准能量 (Ha)
E0 = -1.01546825  |  激发能 = 0.0000 eV
E1 = -0.87542794  |  激发能 = 3.8107 eV
E2 = -0.42938376  |  激发能 = 15.9482 eV
E3 = -0.26922131  |  激发能 = 20.3064 eV


以上内容不要修改

In [2]:
# ==============================================================================
# 3. 扩展哈密顿量（占位符，用于通过 VMC 类型检查）
# ==============================================================================
class ExtendedHamiltonian(nk.operator.DiscreteOperator):
    def __init__(self, base_hamiltonian, K):
        self.base_ham = base_hamiltonian
        self.K = K
        self._hilbert = nk.hilbert.TensorHilbert(*[base_hamiltonian.hilbert] * K)
        super().__init__(self._hilbert)

    @property
    def dtype(self):
        return self.base_ham.dtype

    @property
    def max_conn_size(self) -> int:
        return self.base_ham.max_conn_size * self.K

    def get_conn_padded(self, x):
        n_conn = self.max_conn_size
        xp = jnp.zeros((n_conn, self.hilbert.size), dtype=x.dtype)
        mels = jnp.zeros((n_conn,), dtype=self.dtype)
        return xp, mels

    def get_conn(self, x):
        return self.get_conn_padded(x)

ha_ext = ExtendedHamiltonian(ha, K)


In [3]:
# ==============================================================================
# 4. Linen 模型定义
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 16

    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex64)
        h = nn.tanh(nn.Dense(self.hidden_dim, dtype=jnp.complex64)(x))
        h = nn.tanh(nn.Dense(self.hidden_dim, dtype=jnp.complex64)(h))
        out = nn.Dense(1, dtype=jnp.complex64)(h)
        return out.squeeze()

class NESModel(nn.Module):
    n_spin_orbitals: int
    K: int
    hidden_dim: int = 16

    def setup(self):
        # 创建 K 个独立的单粒子 ansatz
        self.singles = [SingleStateAnsatz(self.hidden_dim) for _ in range(self.K)]

    def __call__(self, x_batch):
        # x_batch: (batch, K * n_spin) -> log ψ
        batch_size = x_batch.shape[0]
        x_reshaped = x_batch.reshape(batch_size, self.K, self.n_spin_orbitals)

        def log_psi_per_sample(x_sample):
            M = jnp.array([
                [self.singles[j](x_sample[i]) for j in range(self.K)]
                for i in range(self.K)
            ])
            det = jnp.linalg.det(M)
            return jnp.log(det + 1e-10)

        return jax.vmap(log_psi_per_sample)(x_reshaped)


In [4]:
# ==============================================================================
# 5. 局域能量计算（使用 bind 避免手动参数解包）
# ==============================================================================
def Ham_psi(ham, single_module_bound, x):
    """计算 (H ψ)(x)，single_module_bound 是已绑定参数的子模块"""
    x_primes, mels = ham.get_conn_padded(x)
    psi_vals = jax.vmap(lambda xp: single_module_bound(xp))(x_primes)
    return jnp.sum(mels * psi_vals)

def compute_local_energy_single(ham, model, params, x_single):
    n_spin = model.n_spin_orbitals
    x_batch = x_single.reshape(model.K, n_spin)

    # 构建 M 矩阵：使用 model.apply 并传入 method='eval_single' 或自定义函数
    # 注意：这里我们直接使用 model.apply 并传入一个额外的索引参数来调用第 j 个单粒子 ansatz
    def psi_j(x, j):
        # 调用 model 的第 j 个 singles 子模块
        return model.apply(params, x, method=lambda m, x: m.singles[j](x))
    
    M = jnp.array([
        [psi_j(x_batch[i], j) for j in range(model.K)]
        for i in range(model.K)
    ])

    # 计算 H_M 矩阵
    H_mat = []
    for i in range(model.K):
        row = []
        for j in range(model.K):
            # 作用哈密顿量
            row.append(Ham_psi(ham, lambda x: psi_j(x, j), x_batch[i]))
        H_mat.append(row)
    Hp = jnp.array(H_mat, dtype=jnp.complex64)

    M_reg = M + 1e-8 * jnp.eye(model.K, dtype=M.dtype)
    E_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(E_mat).real


# 批量版本
batch_local_energy = jax.vmap(compute_local_energy_single, in_axes=(None, None, None, 0))

In [5]:
# ==============================================================================
# 6. 自定义 MCState（重写 expect_and_grad）
# ==============================================================================
class NESMCState(nk.vqs.MCState):
    def expect_and_grad(self, O, *, mutable=None, use_covariance=None):
        print(">>> NESMCState.expect_and_grad called")
        print(self.samples.shape)
        samples = self.samples
        # flatten chains and length dimensions
        samples_flat = samples.reshape(-1, samples.shape[-1])
        model = self.model
        params = self.parameters

        local_energies = batch_local_energy(ha, model, params, samples_flat)
        loss = local_energies.mean()
        print(f"   Loss (trace avg): {loss:.6f}")

        def loss_fn(p):
            les = batch_local_energy(ha, model, p, samples_flat)
            return les.mean()

        grad = jax.grad(loss_fn)(params)
        loss_stats = nk.stats.Stats(mean=loss, variance=0.0, error_of_mean=0.0)
        return loss_stats, grad

# ==============================================================================
# 7. 初始化模型与变分态
# ==============================================================================
model = NESModel(n_spin_orbitals=4, K=K, hidden_dim=16)

# 用虚拟输入初始化参数
dummy_input = jnp.zeros((1, hi_ext.size), dtype=jnp.int8)
params = model.init(jax.random.PRNGKey(42), dummy_input)

vstate = NESMCState(
    sampler,
    model,
    n_samples=1024,
    n_discard_per_chain=16,
    seed=123,
    variables=params
)


In [6]:
# ==============================================================================
# 8. 优化器与 VMC 驱动
# ==============================================================================
optimizer = nk.optimizer.Sgd(learning_rate=0.001)

gs = nk.driver.VMC(
    ha_ext,            # 传入扩展哈密顿量以满足类型检查
    optimizer,
    variational_state=vstate
)

print("\n开始训练...")
gs.run(n_iter=300)     # 正式训练可增加步数


开始训练...


  0%|          | 0/300 [00:00<?, ?it/s]

>>> NESMCState.expect_and_grad called


  0%|          | 0/300 [00:00<?, ?it/s]

(16, 64, 8)


ScopeCollectionNotFound: Tried to access "kernel" from collection "params" in "/singles_0/Dense_0" but the collection is empty. (https://flax.readthedocs.io/en/latest/api_reference/flax.errors.html#flax.errors.ScopeCollectionNotFound)